# 01 — Interactive Data Correction

Use this notebook to investigate a data-quality issue (a meter that suddenly
spikes, a meter that crashes to zero, a day where readings cancel out) and
fix it at the source by adding an entry to `backend/data/corrections.json`.

**Workflow:**

1. **Investigate** — load the cache, run three `find_*` queries
2. **Confirm** — pick a meter, look at its source Excel values, decide
3. **Apply** — `add_correction()` writes to `corrections.json`
4. **Rebuild** — `rebuild_downstream()` re-derives the 12 daily JSONs (~5s)
5. **Verify** — re-run the find queries, confirm the spike is gone

After this notebook, run `bat\real\start_dashboard_real.bat` to refresh the UI.

In [ ]:
# ── Cell 0: imports ────────────────────────────────────────
import sys
from pathlib import Path

# Make the helper importable when the notebook is run from any cwd.
NB_DIR = Path.cwd()
if not (NB_DIR / "_corrections_helper.py").exists():
    NB_DIR = Path(r"C:\Users\Administrator\.openclaw\workspace\portfolio\scripts\notebooks")
sys.path.insert(0, str(NB_DIR))

import _corrections_helper as h  # noqa: E402
import pandas as pd  # noqa: E402

pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 200)
print(f"helper loaded from: {NB_DIR}")
print(f"cache  : {h.CACHE_PATH}")
print(f"output : {h.OUTPUT_DIR}")
print(f"corr   : {h.CORRECTIONS_PATH}")

In [ ]:
# ── Cell 1: Investigate ────────────────────────────────────
# Load the converter's internal cache and run the three find_* queries.
df = h.load_cache_as_df()
if df.empty:
    raise SystemExit("cache is empty — run bat/real/convert_real_data.bat first")

print(f"cache: {len(df):,} rows, {df['meterId'].nunique():,} meters, "
      f"{df['date'].min().date()} → {df['date'].max().date()}")
print()

# Tweak the thresholds here for more/less sensitivity.
outliers = h.find_per_meter_outliers(df, threshold_z=4.0, min_history=14)
jumps    = h.find_daily_jumps(df, threshold_ratio=10.0, min_history=7)
pairs    = h.find_negative_pairs(df)

print(f"per_meter_outliers : {len(outliers):,}")
print(f"daily_jumps        : {len(jumps):,}")
print(f"negative_pairs     : {len(pairs):,}")
print()
print("Top 10 per-meter z-score outliers (most extreme first):")
print(outliers.head(10).to_string(index=False))

In [ ]:
# ── Cell 2: Investigate continued ──────────────────────────
print("Top 10 daily jumps (highest ratio first):")
print(jumps.head(10).to_string(index=False))
print()
print("Top 5 negative-pair candidates (showing date/meterId/value, max 5):")
print(pairs.head(5).to_string(index=False))

In [ ]:
# ── Cell 3: Confirm ─────────────────────────────────────────
# Set the meterId of the suspect spike here. Default: 712720 (the
# historical 4/16-4/27 ×10 config error from the README).
SUSPECT_METER = "712720"
SUSPECT_START = "2026-04-16"
SUSPECT_END   = "2026-04-27"

m = df[df["meterId"] == SUSPECT_METER].sort_values("date")
mask = (m["date"] >= pd.Timestamp(SUSPECT_START)) & (m["date"] <= pd.Timestamp(SUSPECT_END))
print(f"Meter {SUSPECT_METER} — full history ({len(m)} days):")
print(m[["date", "total"]].to_string(index=False))
print()
print(f"Window [{SUSPECT_START}..{SUSPECT_END}] — candidate for correction:")
print(m[mask][["date", "total"]].to_string(index=False))

In [ ]:
# ── Cell 4: Apply ──────────────────────────────────────────
# Set the correction parameters. Defaults reproduce the 712720 fix.
FACTOR  = 0.1   # ÷10 (set to 1.0 for no change)
REASON  = "设置错误-读数×10 (interactive notebook fix)"

# Safety net: this is a no-op if the exact correction already exists
# (the helper checks for overlapping entries).
try:
    new_entry = h.add_correction(
        meterId=SUSPECT_METER,
        start=SUSPECT_START,
        end=SUSPECT_END,
        factor=FACTOR,
        reason=REASON,
    )
    print(f"OK Added: {new_entry}")
except ValueError as e:
    print(f"(skipped: {e})")
    print("Current corrections.json contents:")
    for c in h.load_corrections():
        print(f"  {c}")

In [ ]:
# ── Cell 5: Rebuild ────────────────────────────────────────
# Re-derive the 12 daily JSONs from the (patched) cache. Skips the
# hourly JSONs (those are append-only and require re-reading Excel).
result = h.rebuild_downstream(
    affected_meterIds=[SUSPECT_METER],
    affected_dates=[SUSPECT_START, SUSPECT_END],
)
print(f"rebuild status : {result['status']}")
print(f"files_written  : {len(result.get('files_written', []))}")
for f in result.get("files_written", []):
    print(f"  - {f}")
print(f"hourly_rows_pruned: {result.get('hourly_rows_pruned', 0):,}")
print()
print("Next: run `bat\\real\\start_dashboard_real.bat` to refresh the UI.")
print("      The new converter run with --since will re-insert the hourly")
print("      rows for the corrected window with the /10 factor applied.")

In [ ]:
# ── Cell 6: Verify ─────────────────────────────────────────
# Re-run the find queries on the patched cache. The suspect meter
# should now have lower or no entries in outliers/jumps/pairs.
df2 = h.load_cache_as_df()
outliers2 = h.find_per_meter_outliers(df2, threshold_z=4.0, min_history=14)
jumps2    = h.find_daily_jumps(df2, threshold_ratio=10.0, min_history=7)
pairs2    = h.find_negative_pairs(df2)

n_o = (outliers2["meterId"] == SUSPECT_METER).sum()
n_j = (jumps2["meterId"] == SUSPECT_METER).sum()
n_p = (pairs2["meterId"] == SUSPECT_METER).sum()

print(f"AFTER fix: {SUSPECT_METER} appears in")
print(f"  per_meter_outliers : {n_o} (was ?)")
print(f"  daily_jumps        : {n_j} (was ?)")
print(f"  negative_pairs     : {n_p} (was ?)")
print()
print(f"Meter {SUSPECT_METER} — {SUSPECT_START} to {SUSPECT_END} after fix:")
m2 = df2[df2["meterId"] == SUSPECT_METER].sort_values("date")
mask2 = (m2["date"] >= pd.Timestamp(SUSPECT_START)) & (m2["date"] <= pd.Timestamp(SUSPECT_END))
print(m2[mask2][["date", "total"]].to_string(index=False))

## Notes

- The notebook writes to `backend/data/corrections.json`, which is committed
  to git. Future converter runs will automatically apply the correction
  at the row level (before the cap check).
- The hourly SQLite rows for the corrected window are **pruned** in this
  notebook so the next `convert_real_data.bat --since <start>` run
  re-inserts them with the corrected values (`INSERT OR IGNORE` would
  otherwise leave the old wrong values in place).
- The dashboard HTML is not rebuilt by this notebook — run
  `bat\real\start_dashboard_real.bat` to refresh it.